In [ ]:
netflix_raw_df = (
    spark.read
    .format("csv")
    .option("header",True)
    .option("quote",'"')
    .option("escape",'"')
    .option("multiLine", True)
    .option("inferSchema",True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/netflix_titles.csv")
)
netflix_raw_df.display()

In [ ]:
print(netflix_raw_df.select("show_id").count())
print(netflix_raw_df.select("show_id").distinct().count())

In [ ]:
print(netflix_raw_df.select("title").count())
print(netflix_raw_df.select("title").distinct().count())

In [ ]:
from pyspark.sql.functions import col
netflix_raw_df.groupBy("title").count().filter(col("count")>1).display()

In [ ]:
netflix_raw_df.printSchema()

In [ ]:
netflix_raw_df.groupBy("type").count().display()

In [ ]:
netflix_raw_df.groupBy("country").count().display()

In [ ]:
netflix_raw_df.groupBy("release_year").count().display()

In [ ]:
netflix_raw_df.groupBy("duration").count().display()

In [ ]:
netflix_raw_df.filter(col("duration").isNull()).count()

In [ ]:
netflix_raw_df.select("date_added").distinct().display()

In [ ]:
netflix_raw_df.groupBy("rating").count().display()

In [ ]:
netflix_raw_df.groupBy("listed_in").count().display()

In [ ]:
netflix_raw_df.show(5)

In [ ]:
from pyspark.sql.functions import col, trim, lower, regexp_replace

netflix_raw_df = netflix_raw_df.withColumns({
    "type":trim(col("type")),
    "title":trim(col("title")),
    "director":trim(col("director")),
    "cast":trim(col("cast")),
    "country":(trim(regexp_replace(col("country"),"^,",""))),
    "date_added":trim(col("date_added")),
    "release_year":lower(trim(col("release_year"))),
    "rating":trim(col("rating")),
    "duration":lower(trim(col("duration"))),
    "listed_in":trim(col("listed_in")),
    "description":trim(col("description"))
  
})
netflix_raw_df.display()

In [ ]:
netflix_raw_df.display()

In [ ]:
netflix_raw_df.groupBy("release_year").count().display()

In [ ]:
netflix_raw_df.groupBy("country").count().display()

In [ ]:
netflix_raw_df.display()

In [ ]:
from pyspark.sql.functions import col, contains, when, lower

netflix_clean_raw_df = netflix_raw_df.withColumns({
    "duration_min": when(lower(col("duration")).contains("min"), col("duration")).otherwise(None),
    "duration_season": when(lower(col("duration")).contains("season"), col("duration")).otherwise(None),
    "rating": when(lower(col("rating")).contains("min"), None).otherwise(col("rating"))
})
netflix_clean_raw_df.display()

In [ ]:
netflix_clean_raw_df = netflix_clean_raw_df.drop("duration")
netflix_clean_raw_df.display()

In [ ]:
from pyspark.sql.types import StringType, DateType, LongType
from pyspark.sql.functions import col, split, to_date

netflix_silver_df = netflix_clean_raw_df.withColumns({
    "show_id": col("show_id").cast(StringType()),
    "type": col("type").cast(StringType()),
    "title": col("title").cast(StringType()),
    "director": col("director").cast(StringType()),
    "cast": col("cast").cast(StringType()),
    "country": col("country").cast(StringType()),
    "date_added": to_date(col("date_added"), "MMMM d, yyyy"),
    "release_year": col("release_year").cast(LongType()),
    "rating": col("rating").cast(StringType()),
    "duration_min": split(col("duration_min"), " ")[0].cast(LongType()),
    "duration_season": split(col("duration_season"), " ")[0].cast(LongType()),
    "listed_in": col("listed_in").cast(StringType()),
    "description": col("description").cast(StringType())
})
netflix_silver_df.display()

In [ ]:
netflix_silver_df.printSchema()

In [ ]:
netflix_silver_df.write.mode("overwrite").saveAsTable("dev.mini_projects.netflix_silver")

In [ ]:
spark.read.table("dev.mini_projects.netflix_silver").display()